In [2]:
import pandas as pd
import gpxpy
import os
import re

## Configuration
Remplacez par le chemin vers les fichiers à traiter

In [13]:
data_path = './input/AE_HO_008_2025-10-20-14_53_13_raw.csv'
track_source = './input/traces'  # Dossier contenant les fichiers .gpx (ou chemin direct vers un fichier .gpx)
instrument_name = 'AE_HO_008' # Suivre le format du numéro de série (owner_type_numéro)
output_dir = "output"
csv_encoding = 'utf-8' # or 'latin-1' depending on your file encoding

## Étape 1 : Charger les données GPX

In [14]:
def load_gpx_tracks(path):
    if os.path.isdir(path):
        gpx_files = sorted(
            os.path.join(path, file_name)
            for file_name in os.listdir(path)
            if file_name.lower().endswith('.gpx')
        )
    else:
        gpx_files = [path]
    
    tracks = []
    for gpx_file_path in gpx_files:
        with open(gpx_file_path, 'r') as gpx_file:
            gpx = gpxpy.parse(gpx_file)
        
        for track in gpx.tracks:
            points = []
            for segment in track.segments:
                for point in segment.points:
                    points.append({
                        'datetime': point.time,
                        'lat': point.latitude,
                        'long': point.longitude
                    })
            if points:
                df_gpx = pd.DataFrame(points)
                df_gpx['datetime'] = pd.to_datetime(df_gpx['datetime']).dt.tz_convert('UTC')
                df_gpx = df_gpx.set_index('datetime')
                tracks.append(df_gpx)
    return tracks

gpx_tracks = load_gpx_tracks(track_source)

print(f"Nombre de traces chargées : {len(gpx_tracks)}")

# Afficher la plage de temps de chaque tracée
for i, df_gpx in enumerate(gpx_tracks):
    print(f"Tracée {i+1} : de {df_gpx.index.min()} à {df_gpx.index.max()}")

Nombre de traces chargées : 17
Tracée 1 : de 2025-08-04 07:19:05.134000+00:00 à 2025-08-04 16:16:25.125000+00:00
Tracée 2 : de 2025-08-08 17:16:03.946000+00:00 à 2025-08-08 20:54:44.965000+00:00
Tracée 3 : de 2025-08-11 07:24:41.861000+00:00 à 2025-08-11 17:17:43.832000+00:00
Tracée 4 : de 2025-08-12 12:23:24.062000+00:00 à 2025-08-13 06:51:54.169000+00:00
Tracée 5 : de 2025-08-16 08:14:59.844000+00:00 à 2025-08-16 15:59:50.805000+00:00
Tracée 6 : de 2025-08-17 07:16:12.047000+00:00 à 2025-08-17 22:13:29.013000+00:00
Tracée 7 : de 2025-08-18 06:47:19.921000+00:00 à 2025-08-18 15:52:54.930000+00:00
Tracée 8 : de 2025-08-05 17:27:20.247000+00:00 à 2025-08-05 20:22:27.273000+00:00
Tracée 9 : de 2025-08-10 07:04:40.774000+00:00 à 2025-08-10 17:07:35.821000+00:00
Tracée 10 : de 2025-08-02 08:41:55.291000+00:00 à 2025-08-03 06:17:12.399000+00:00
Tracée 11 : de 2025-08-03 07:13:11.064000+00:00 à 2025-08-03 16:27:31.005000+00:00
Tracée 12 : de 2025-08-06 08:16:30.189000+00:00 à 2025-08-06 16:4

## Étape 2 : Charger les données CSV

In [15]:
def rename_columns_with_regex(df):
    # Mapping with regex patterns (the number is replaced by \\d+)
    col_map = {
        r'Température.*': 'sea_temp',
        r'Pression absolue.*': 'sea_pres',
        r'Conductivité électrique.*': 'ec_abs',
        r'Conductivité spécifique.*': 'ec25',
        r'Salinité.*': 'sea_sal',
        r'Matières solides dissoutes totales.*': 'total_dissolved_solids',
    }
    new_columns = {}
    for col in df.columns:
        new_col = col
        for pattern, replacement in col_map.items():
            if re.fullmatch(pattern, col):
                new_col = replacement
                break
        new_columns[col] = new_col
    return df.rename(columns=new_columns)

df = pd.read_csv(data_path, low_memory=False, parse_dates=[1], date_format="%m/%d/%Y %H:%M:%S", delimiter=';', encoding=csv_encoding)

# Renommer les colonnes avec regex
# La colonne 'Date et heure (CEST)' reste renommée explicitement
if 'Date et heure (CEST)' in df.columns:
    df = df.rename(columns={'Date et heure (CEST)': 'datetime'})
df = rename_columns_with_regex(df)

# Supprimer les colonnes inutiles
df = df.drop(
    columns=['#', 'Hôte connecté', 'Fin du fichier', 'Démarré', 'Arrêté', 'Bouton Haut', 'Bouton Bas', 'Détection d’eau'],
    errors='ignore'
)

print(df.head())

# Convertir le fuseau horaire de CEST en UTC
df['datetime'] = df['datetime'].dt.tz_localize('Europe/Paris')
df['datetime'] = df['datetime'].dt.tz_convert('UTC')
df = df.set_index('datetime')

             datetime sea_temp sea_pres   ec_abs     ec25 sea_sal  \
0 2025-07-21 18:44:07    11,35  102,302  37570,4    52672   33,22   
1 2025-07-21 18:45:07    11,31  102,301  37390,5  52471,4   33,07   
2 2025-07-21 18:46:07    11,28    102,3  37427,3  52570,2   33,14   
3 2025-07-21 18:47:07    11,26   102,34  37345,7    52498   33,08   
4 2025-07-21 18:48:07    11,23  102,299  37246,3  52400,7   33,01   

  total_dissolved_solids  
0               24420,77  
1                24303,8  
2               24327,73  
3               24274,69  
4               24210,11  


In [16]:
print("Plage de dates CSV :", df.index.min(), "à", df.index.max())

Plage de dates CSV : 2025-07-21 16:44:07+00:00 à 2025-10-20 13:51:49+00:00


## Étape 3 : Interpolation de la position des mesures à partir des traces GPS  

In [17]:
def interpolate_position(row, df_gpx):
    t = row.name
    idx = df_gpx.index.searchsorted(t, side='right') - 1
    if idx < 0 or idx >= len(df_gpx) - 1:
        return None, None  # Hors des limites
    point_A = df_gpx.iloc[idx]
    point_B = df_gpx.iloc[idx + 1]
    tA, tB = point_A.name, point_B.name
    latA, longA = point_A['lat'], point_A['long']
    latB, longB = point_B['lat'], point_B['long']
    ratio = (t - tA).total_seconds() / (tB - tA).total_seconds()
    lat = latA + (latB - latA) * ratio
    long = longA + (longB - longA) * ratio
    return lat, long

def interpolate_all_tracks(df, gpx_tracks):
    dfs = []
    for df_gpx in gpx_tracks:
        # Filtrer les mesures dans la plage de la tracée
        mask = (df.index >= df_gpx.index.min()) & (df.index <= df_gpx.index.max())
        df_sub = df[mask].copy()
        if df_sub.empty:
            continue
        df_sub[['lat', 'long']] = df_sub.apply(lambda row: interpolate_position(row, df_gpx), axis=1, result_type='expand')
        df_sub = df_sub.dropna(subset=['lat', 'long'])
        dfs.append(df_sub)
    if dfs:
        return pd.concat(dfs).sort_index()
    else:
        return pd.DataFrame()

In [18]:
df_interp = interpolate_all_tracks(df, gpx_tracks)

## Étape 4 : Exporter les données

In [19]:
os.makedirs(output_dir, exist_ok=True)

if not df_interp.empty:
    start_at = df_interp.index.min().strftime('%Y-%m-%d_%H_%M_%S')
    output_csv_final = os.path.join(output_dir, f"{instrument_name}_{start_at}_with_positions.csv")
    df_interp.to_csv(output_csv_final, sep=';', date_format='%Y-%m-%dT%H:%M:%SZ')
    print(f"Fichier exporté : {output_csv_final}")
else:
    print("Aucune donnée interpolée à exporter.")

Fichier exporté : output/AE_HO_008_2025-08-02_08_42_49_with_positions.csv
